<a href="https://colab.research.google.com/github/somraj112/TELECOM_CUSTOMER_CHURN_PREDICTION/blob/main/churn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **TELECOM CUSTOMER CHURN PREDICTION**

### Customer Churn

Customer churn occurs when customers stop using a company’s service. In the telecom industry, customers frequently switch providers, resulting in high annual churn rates of 15–25%. Since retaining existing customers costs less than acquiring new ones, companies focus on predicting which customers are likely to leave. By analyzing customer behavior and interactions, businesses can target high-risk customers with effective retention strategies, helping reduce losses and increase profitability.

### Objectives
I will explore the data and try to answer some questions like:

- What's the % of Churn Customers and customers that keep in with the active services?
- Is there any patterns in Churn Customers based on the gender?
- Is there any patterns/preference in Churn Customers based on the type of service provided?
- What's the most profitable service types?
- Which features and services are most profitable?
- Many more questions that will arise during the analysis

## Importing Libraries and Importing Dataset

### Importing Libraries

Loading all necessary libraries

In [ ]:
import sys

!{sys.executable} -m pip install pandas numpy matplotlib seaborn plotly scikit-learn langchain langchain-groq chromadb langchain-core langchain-community sentence-transformers langgraph -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 103.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.7/

In [ ]:
import pandas as pd
import numpy as np
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

### Importing Dataset

Importing the dataset from Kaggle and converting it into a pandas DataFrame for analysis and preprocessing.

In [ ]:
df = pd.read_csv('telco_dataset.csv')

# Exploratory Data Analysis (EDA)

In EDA, we will explore and understand the dataset by analyzing patterns, distributions, relationships, and detecting any missing or unusual values before building models.

In [ ]:
# Loading top 5 rows
df.head()

In [ ]:
# Loading bottom 5 rows
df.tail()

The data set includes information about:

- Customers who left within the last month – the column is called Churn

- Services that each customer has signed up for – phone, multiple lines, internet, online security, online backup, device protection, tech support, and streaming TV and movies

- Customer account information - how long they’ve been a customer, contract, payment method, paperless billing, monthly charges, and total charges

- Demographic info about customers – gender, age range, and if they have partners and dependents

In [ ]:
# Dimention of Dataset
df.shape

In [ ]:
# Info about the dataset
df.info()

In [ ]:
# Array of all columns in Dataset
df.columns.values

In [ ]:
# Showing all column Types
df.dtypes

The target the we will use to guide the exploration is Churn

## Visualize missing values

In [ ]:
# Visualize missing values as a matrix
msno.matrix(df);

Using this matrix we can very quickly find the pattern of missingness in the dataset.

- From the above visualisation we can observe that it has no peculiar pattern that stands out. In fact there is no missing data.

## Data Manipulation

In [ ]:
# Removing Unnecessary Columns, which is unnecessary for model training
df = df.drop(['customerID'], axis = 1)
df.head()

On deep analysis, we can find some indirect missingness in our data (which can be in form of blankspaces)

In [ ]:
# Convert TotalCharges to numeric (invalid values become NaN) and check missing values in each column
df['TotalCharges'] = pd.to_numeric(df.TotalCharges, errors='coerce')
df.isnull().sum()

Here we see that the TotalCharges has 11 missing values. Let's check this data.

In [ ]:
# Showing rows where TotalCharges is NaN (missing)
df[np.isnan(df['TotalCharges'])]

It can also be noted that the Tenure column is 0 for these entries even though the MonthlyCharges column is not empty.

In [ ]:
# Get index numbers of rows where tenure is 0
df[df['tenure'] == 0].index

There are no additional missing values in the Tenure column

In [ ]:
# Remove rows where tenure is 0 and verify they are deleted
df.drop(labels=df[df['tenure'] == 0].index, axis=0, inplace=True)
df[df['tenure'] == 0].index

In [ ]:
df.fillna(df["TotalCharges"].mean())

In [ ]:
df.isnull().sum()

In [ ]:
df['Churn'].value_counts()

In [ ]:
sns.countplot(x='Churn', data=df)
plt.title("Churn Distribution")
plt.show()

In [ ]:
# Select numerical columns
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Plot boxplots for each numerical feature vs Churn
for col in num_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(x='Churn', y=col, data=df)
    plt.title(f"{col} vs Churn")
    plt.show()

In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df[['tenure','MonthlyCharges','TotalCharges','Churn']].corr(),
            annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Correlation Matrix")
plt.show()

## Encoding Categorical Variables
Machine learning models cannot work with categorical (text) data.
Therefore, we convert all categorical features into numerical format
using One-Hot Encoding.

We use drop_first=True to avoid multicollinearity
(dummy variable trap).

In [ ]:
 # Check categorical columns
df.select_dtypes(include='object').columns

In [ ]:
# Apply one-hot encoding
df = pd.get_dummies(df, drop_first=True)

df.head()

## Splitting Features and Target Variable
In this step, we separate the independent variables (X)
from the dependent variable (y).

X → All input features  
y → Target variable (Churn)

This separation is necessary before training a machine learning model.

In [ ]:
# Separating features and target
X = df.drop("Churn", axis=1)
y = df["Churn"]

# Checking shapes
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

## Feature Scaling
Feature scaling standardizes the range of numerical features.

Since different features have different scales
(e.g., tenure vs TotalCharges),
we apply StandardScaler to bring them to the same scale.

This improves the performance of distance-based
and gradient-based algorithms.

In [ ]:
# Initializing scaler
scaler = StandardScaler()

# Fitting and transforming feature set
X_scaled = scaler.fit_transform(X)

## Train-Test Split
We split the dataset into training and testing sets.

Training set → Used to train the model  
Testing set → Used to evaluate model performance

We use an 80-20 split and set random_state=42
to ensure reproducibility.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

## Logistic Regression Model
Logistic Regression is a supervised machine learning algorithm
used for binary classification problems.

Since Customer Churn prediction is a binary problem
(Churn = Yes or No), Logistic Regression is a strong baseline model.

It also provides probability outputs and is easy to interpret.

In [ ]:
# Initializing the model
lr = LogisticRegression()

# Training the model
lr.fit(X_train, y_train)

In [ ]:
y_pred_lr = lr.predict(X_test)

## Model Evaluation
We evaluate the model using multiple performance metrics.

Accuracy alone is not sufficient because the dataset may be imbalanced.

Therefore, we evaluate using:
- Accuracy
- Confusion Matrix
- Precision
- Recall
- F1-Score


In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_lr))

In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

In [ ]:
# Create confusion matrix
cm = confusion_matrix(y_test, y_pred_lr)

# Plot heatmap
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Confusion Matrix - Logistic Regression")

plt.show()

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred_lr))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

y_prob = lr.predict_proba(X_test)[:, 1]

# Add slight noise to spread actual values
y_test_jitter = y_test + np.random.normal(0, 0.02, size=len(y_test))

plt.figure(figsize=(10, 6))

plt.scatter(y_test_jitter, y_prob, alpha=0.5)

plt.plot([0,1], [0,1], 'r--', lw=2)

plt.xlabel('Actual Churn')
plt.ylabel('Predicted Probability')
plt.title('Actual vs Predicted Probability (Logistic Regression)')

plt.show()

## Decision Tree Model


In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Initialize model
dt = DecisionTreeClassifier(random_state=42)

# Train model
dt.fit(X_train, y_train)

# Predict
y_pred_dt = dt.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_dt))

print("Classification Report:")
print(classification_report(y_test, y_pred_dt))

In [ ]:
# Generate confusion matrix
cm_dt = confusion_matrix(y_test, y_pred_dt)

# Plot heatmap
plt.figure(figsize=(6,5))
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Greens')

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Confusion Matrix - Decision Tree")

plt.show()

## Model Comparison
In this step, we compare the performance of Logistic Regression
and Decision Tree models using multiple evaluation metrics.

We focus not only on accuracy, but also on recall,
F1-score, and ROC-AUC score since churn prediction
is an imbalanced classification problem.

In [ ]:
# Logistic Regression Metrics
lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_roc = roc_auc_score(y_test, lr.predict_proba(X_test)[:,1])

# Decision Tree Metrics
dt_accuracy = accuracy_score(y_test, y_pred_dt)
dt_roc = roc_auc_score(y_test, dt.predict_proba(X_test)[:,1])
print("Logistic Regression Accuracy:", lr_accuracy)
print("Decision Tree Accuracy:", dt_accuracy)


In [ ]:
from imblearn.over_sampling import SMOTE

# Initialize SMOTE
smote = SMOTE(random_state=42)

# Apply SMOTE only on training data
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Check class distribution
print("Before SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(pd.Series(y_train_smote).value_counts())

In [ ]:
from sklearn.linear_model import LogisticRegression

# Train model on SMOTE data
lr_smote = LogisticRegression(max_iter=1000)

lr_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = lr_smote.predict(X_test)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_smote))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_smote))
# Predict on original test data
y_pred_smote = lr_smote.predict(X_test)

## Saving the model

In [ ]:
import joblib

# Saved trained model
joblib.dump(lr, "logistic_model.pkl")

# Saved scaler
joblib.dump(scaler, "scaler.pkl")

# Saved feature column order
joblib.dump(X.columns, "feature_columns.pkl")

print("Model, Scaler and Columns Saved Successfully!")

# End Term Evaluation

## Load Model Components

This step loads the saved model and preprocessing components required for prediction.

* Load trained Logistic Regression model
* Load scaler for feature normalization
* Load feature column order
* Ensures consistent input format for predictions

In [ ]:
# This section loads the trained ML model, scaler,
# and feature column order saved during training.

# Load trained Logistic Regression model
model = joblib.load("logistic_model.pkl")

# Load scaler used during preprocessing
scaler = joblib.load("scaler.pkl")

# Load feature column order (important for consistency)
feature_columns = joblib.load("feature_columns.pkl")

print(" All components loaded successfully!")

## Churn Prediction Function

This function takes user input, processes it using the same steps as training, and predicts the probability of customer churn.

* Converts input data into DataFrame
* Aligns columns with training format
* Applies feature scaling
* Predicts churn probability using model
* Classifies risk as High, Medium, or Low
* Returns structured prediction output

In [ ]:
def predict_churn(input_data: dict):

    # Convert input dictionary to DataFrame
    df = pd.DataFrame([input_data])

    # Ensure same column order as used during training
    # Missing columns are filled with 0
    df = df.reindex(columns=feature_columns, fill_value=0)

    # Apply scaling
    scaled_data = scaler.transform(df)

    # Predict churn probability
    prob = model.predict_proba(scaled_data)[0][1]

    # Categorize risk level based on probability
    if prob > 0.7:
        risk = "HIGH"
    elif prob > 0.4:
        risk = "MEDIUM"
    else:
        risk = "LOW"

    # Return structured output
    return {
        "churn_probability": float(prob),
        "risk_level": risk
    }

## Test With Sample Customer

This step tests the trained model using a sample customer input to verify prediction output.

* Define sample customer features
* Pass input to prediction function
* Generate churn probability and risk level
* Display prediction result for validation

In [ ]:
sample_customer = {
    "tenure": 2,
    "MonthlyCharges": 80,
    "TotalCharges": 160,
    # Add all required features here
}

# Run prediction
result = predict_churn(sample_customer)

# Display output
print("Prediction Result:")
print(result)

## Initialize Groq LLM

This step sets up the Groq LLM to enable intelligent reasoning and future agent-based workflows.

* Import required libraries
* Set Groq API key
* Initialize ChatGroq model
* Configure model parameters (temperature, model type)
* Prepares system for LLM-based retention strategies

In [ ]:
import os
from langchain_groq import ChatGroq

# Set your Groq API key
groq_api_key = os.getenv("GROQ_API_KEY")

# Initialize model (fast + good)
llm = ChatGroq(
    model="llama-3.3-70b-versatile",   # strong model
    temperature=0.3
)

print("Groq LLM initialized successfully!")

## Customer Risk Analysis Function

his function uses the LLM to interpret churn predictions and explain customer risk in business terms.

* Takes customer data and prediction as input
* Creates a structured prompt for analysis
* Uses Groq LLM for reasoning
* Identifies key churn drivers and patterns
* Returns clear, concise, actionable insights

In [ ]:
def analyze_customer(customer_data, prediction):
    """
    Uses Groq LLM to explain churn risk in business terms
    """

    prompt = f"""
    You are a customer retention expert.

    Customer Data:
    {customer_data}

    Churn Prediction:
    {prediction}

    Analyze and explain:
    - Key reasons for churn risk
    - Behavioral or business patterns

    Keep it:
    - Clear
    - Concise
    - Actionable

    Output as bullet points.
    """

    response = llm.invoke(prompt)

    return response.content

## AI-Based Risk Reasoning Output

This step generates and displays LLM-based insights explaining the churn prediction.

* Uses prediction and customer data as input
* Calls analysis function for reasoning
* Produces business-level explanation of churn risk
* Highlights key drivers and patterns
* Displays actionable recommendations for retention

In [ ]:
prediction = predict_churn(sample_customer)

reasoning = analyze_customer(sample_customer, prediction)

print("AI Reasoning:\n")
print(reasoning)

AI Reasoning:

Based on the customer data and churn prediction, here are the key insights:

* **Key reasons for churn risk:**
  * Relatively short tenure of 2 months, indicating the customer is still in the early stages of their contract and may be more prone to switching.
  * Monthly charges of $80, which may be perceived as high, potentially leading to dissatisfaction.
* **Behavioral or business patterns:**
  * The customer has paid a total of $160, suggesting they are currently fulfilling their payment obligations, but may be re-evaluating their contract due to the relatively high monthly charges.
  * The low risk level indicates that the customer is not exhibiting extreme churn behavior, but the 23.95% churn probability suggests that proactive measures should be taken to retain them.
* **Actionable recommendations:**
  * Offer personalized discounts or promotions to reduce the perceived high monthly charges.
  * Provide exceptional customer service to build trust and loyalty.
  * R

## Create Retention Strategy Data